In [1]:
%load_ext autoreload
%autoreload 2

import xarray as xr
import numpy as np
from cop_ard_app.computation import create_client
from cop_ard_app.dem import import_dem, stack_dem
from cop_ard_app.io import Path, normpath, read, glob, delete
from cop_ard_app.area_projection import stack_geometry, stack_mask
from cop_ard_app.simulation import stack_simulation, stack_simulation_ml
from cop_ard_app.slc import (
    import_slc_stack,
    finalize_import,
    extract_orbits,
)
from cop_ard_app.utils import get_bbox
from cop_ard_app.model import update_coreg
from cop_ard_app.etad import import_ETAD
from cop_ard_app.orbit import import_ORBIT
from cop_ard_app.coreg import stack_coregistration
from cop_ard_core.logger import logger
from cop_ard_app.utils import get_bbox

# Reference Image

In [2]:
in_files = glob("/data/SENTINEL1/SURAT/SAFE/*20231218*SAFE")
path = "/data/SENTINEL1/SURAT/NOTEBOOK"
wdir_path_static = f"{path}/STATIC"
wdir_path_tmp = f"{path}/DYNAMIC"
slcs_path = f"{wdir_path_tmp}/slcs"

indices = [0, 1, 27, 28]
chunks_sr = {"acquisition_date": 1, "polarization": 1, "azimuth_time": 2048, "slant_range_time": 2048}
chunks = {"x": 2048, "y": 2048}
chunk_length = max([v for v in chunks.values()])
spacing = {"x": 30, "y": 30}
recompute = True
multiburst = True
do_import = False if (recompute and multiburst) else True

In [3]:
import dask

dask.config.set({"distributed.comm.timeouts.connect": "120s", "distributed.comm.timeouts.tcp": "120s"})
type, n_jobs, job_memory = ["LOCAL", 4, 0]
client = create_client(type, n_jobs, job_memory)
client

2026-08-11 17:09:14 [info     ] <Cluster info: type=LOCAL, n_jobs=4, job_memory=0>


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45581,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35043,Total threads: 1
Dashboard: http://127.0.0.1:41233/status,Memory: 0 B
Nanny: tcp://127.0.0.1:35181,


In [4]:
delete(slcs_path, recursive=True, allow_non_existent=True)
slcs = import_slc_stack(
    in_files,
    slcs_path,
    deramp=True,
    multiburst=multiburst,
    noise_removal_flag=True,
    apply_invalid_mask_flag=True,
    indices=indices,
    calib_type="BetaZero",
    chunks=chunks_sr,
    do_import=do_import,
    enable_dask=False,
)

2026-08-11 17:09:15 [info     ] Importing SLC image structure only (data import skipped).
2026-08-11 17:09:15 [info     ] <Cluster info: type=DUMMY, n_jobs=22, job_memory=2893000368>
2026-08-11 17:09:15 [info     ] <Cluster info: type=DUMMY, n_jobs=22, job_memory=2893000368>
2026-08-11 17:09:16 [info     ] Initializing groups: dict_keys(['bursts/iw1_19156/orig/S1A_51706', 'bursts/iw1_19157/orig/S1A_51706'])
2026-08-11 17:09:17 [info     ] <Cluster info: type=DUMMY, n_jobs=22, job_memory=2893000368>
2026-08-11 17:09:18 [info     ] <Cluster info: type=DUMMY, n_jobs=22, job_memory=2893000368>
2026-08-11 17:09:23 [info     ] Function import_slc_stack took 7.6181 seconds


In [5]:
slcs = xr.DataTree()
if multiburst:
    files = glob(f"{slcs_path}/bursts/*/orig/*")
    for f in files:
        name = f.strip(slcs_path)
        slcs[name] = read(f)
else:
    slcs = read(slcs_path)

In [6]:
slcs, reference_date = finalize_import(slcs=slcs, working_directory=wdir_path_static, recompute=recompute)

2026-08-11 17:09:23 [info     ] Saving reference SLC structure to static working directory.
2026-08-11 17:09:24 [info     ] Function finalize_import took 1.5628 seconds


In [7]:
bbox = get_bbox(slcs)
multiburst = True if "bursts" in slcs else False
dem = import_dem(
    bbox=bbox,
    # dem_type=f"{wdir_path_tmp}/dem",
    working_dir=wdir_path_tmp,
    chunks=chunks,
)

In [8]:
dem_tree = stack_dem(dem, slcs, multiburst, spacing, wdir_path_static, wdir_path_tmp, chunks, recompute=True)

2026-08-11 17:09:54 [warning  ] interp_dem-5a5ec400-8646-4085-8334-4de1bbf66a16 cancelled for reason: unknown., retrying in 1 seconds...
2026-08-11 17:10:09 [info     ] Function stack_dem took 31.6635 seconds


In [9]:
dem_tree = xr.DataTree()
for k, v in (slcs.bursts if multiburst else {"data": slcs}).items():
    path = f"bursts/{k}" if multiburst else "data"
    d = read(f"{wdir_path_tmp}/{path}/dem_resampled")
    if multiburst:
        dem_tree[path] = d
    else:
        dem_tree = xr.DataTree(d)

In [10]:
orbits = extract_orbits(
    slcs,
    multiburst=multiburst,
    acquisition_date=[reference_date],
)

In [11]:
simulation_tree = stack_simulation(
    dem_tree=dem_tree,
    orbits=orbits,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    chunks={**chunks, "acquisition_date": 1},
    master_image=True,
    recompute=recompute,
)

2026-08-11 17:10:22 [info     ] Function stack_simulation took 12.5236 seconds


In [12]:
simulation_ml_tree = stack_simulation_ml(
    simulation_tree=simulation_tree,
    reference_date=reference_date,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    recompute=recompute,
    enable_dask=True,
)

2026-08-11 17:10:23 [info     ] Function stack_simulation_ml took 0.9153 seconds


In [13]:
stack_geometry(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    working_directory_static=wdir_path_static,
    working_directory_tmp=wdir_path_tmp,
    reference_date=reference_date,
    chunk_length=chunk_length,
    recompute=recompute,
    multiburst=multiburst,
    enable_dask=True,
)

2026-08-11 17:10:43 [warning  ] blockwise_rtc_ap-9f242d39-51dd-42e7-b448-f70973061bf2 cancelled for reason: lost dependencies., retrying in 1 seconds...
2026-08-11 17:10:48 [warning  ] blockwise_rtc_ap-71a63688-ee8e-4192-9549-fe2111fb7ee9 cancelled for reason: lost dependencies., retrying in 1 seconds...


In [14]:
stack_mask(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    working_directory_static=wdir_path_static,
    reference_date=reference_date,
    recompute=True,
    multiburst=multiburst,
    enable_dask=True,
)

2026-08-11 17:11:40 [info     ] Function stack_mask took 33.5211 seconds
